# EXO-INSIGHT — Milestone 5: Decision Model and Machine Learning Development

**Project:** EXO-INSIGHT  
**Milestone:** M5  
**Assignment blocks addressed:** Block 4 — A decision model; Block 7 — A model learned from data

## Purpose

This notebook develops and evaluates a machine-learning model that learns to reproduce the **M4 conservative habitable-zone screening label**. It also evaluates how a decision threshold changes false positives and false negatives.

**Scientific boundary:** the target label is derived from the M4 screening protocol. It is not a confirmed habitability label and is not evidence that a planet hosts life. The model's predicted probability is the model's estimated probability of matching that screening label, not the probability of life.

A separate, transparent habitability-potential assessment may be developed only with scientifically justified criteria and explicit limitations. It must not be confused with this supervised classification experiment.

## M5 workflow

1. Locate and load the existing M3 and M4 data without modifying them.
2. Validate the join and inspect the target.
3. Define a leakage-conscious feature set.
4. Split the data into stratified training and test sets.
5. Compare a majority-class baseline with logistic regression.
6. Evaluate performance on held-out data.
7. Inspect decision-threshold trade-offs and errors.
8. Record limitations and the next actions.

**No files are exported by this notebook.**

## 1. Imports and reproducibility

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    precision_recall_curve
)

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
TEST_SIZE = 0.20

print("Libraries imported.")
print("Random state:", RANDOM_STATE)
print("Test size:", TEST_SIZE)

## 2. Locate the existing project data

Expected project files:

- `DATA/processed/exo_insight_preprocessed.csv`
- `DATA/processed/milestone4/m4_ml_training_labels_review.csv`

The notebook searches from the current working directory and its parent folders. If your project is not extracted, extract the existing project ZIP first; do not create another copy of the datasets.

In [ ]:
def find_project_file(filename, start=Path.cwd()):
    search_roots = [start, *start.parents]
    checked = set()

    for root in search_roots:
        if root in checked:
            continue
        checked.add(root)

        matches = list(root.rglob(filename))
        if matches:
            # Prefer a path inside the EXO INSIGHTS project directory.
            matches.sort(key=lambda p: ("EXO INSIGHTS" not in str(p), len(str(p))))
            return matches[0]

    raise FileNotFoundError(
        f"Could not find {filename}.\n"
        "Open this notebook from the extracted EXO INSIGHTS project, "
        "or set the paths manually in this cell."
    )

PREPROCESSED_PATH = find_project_file("exo_insight_preprocessed.csv")
LABELS_PATH = find_project_file("m4_ml_training_labels_review.csv")

print("Preprocessed data:", PREPROCESSED_PATH)
print("M4 labels:", LABELS_PATH)

preprocessed = pd.read_csv(PREPROCESSED_PATH)
labels = pd.read_csv(LABELS_PATH)

print("\nPreprocessed shape:", preprocessed.shape)
print("M4 labels shape:", labels.shape)

## 3. Validate columns and merge M3 features with M4 labels

The merge uses `pl_name` and requires a one-to-one match. A failed validation stops the notebook rather than silently continuing with incomplete or duplicated records.

In [ ]:
required_feature_columns = [
    "pl_name",
    "pl_rade",
    "pl_bmasse",
    "pl_bmassprov",
    "discoverymethod",
    "disc_year"
]

required_label_columns = [
    "pl_name",
    "ml_target_conservative_hz"
]

missing_features = [c for c in required_feature_columns if c not in preprocessed.columns]
missing_labels = [c for c in required_label_columns if c not in labels.columns]

if missing_features or missing_labels:
    raise ValueError(
        f"Missing feature columns: {missing_features}\n"
        f"Missing label columns: {missing_labels}"
    )

if preprocessed["pl_name"].duplicated().any():
    raise ValueError("Duplicate planet names found in M3 data.")

if labels["pl_name"].duplicated().any():
    raise ValueError("Duplicate planet names found in M4 label data.")

data = labels[required_label_columns].merge(
    preprocessed[required_feature_columns],
    on="pl_name",
    how="left",
    validate="one_to_one",
    indicator=True
)

print("Merge results:")
display(data["_merge"].value_counts().rename("records").to_frame())

unmatched = data.loc[data["_merge"] != "both", "pl_name"].tolist()
if unmatched:
    raise ValueError(f"{len(unmatched)} M4 records did not match M3. Examples: {unmatched[:10]}")

data = data.drop(columns="_merge").copy()
data = data.dropna(subset=["ml_target_conservative_hz"]).copy()

data["ml_target_conservative_hz"] = pd.to_numeric(
    data["ml_target_conservative_hz"], errors="coerce"
)
data = data.dropna(subset=["ml_target_conservative_hz"]).copy()

if not set(data["ml_target_conservative_hz"].unique()).issubset({0, 1}):
    raise ValueError("The target must contain only binary values 0 and 1.")

data["ml_target_conservative_hz"] = data["ml_target_conservative_hz"].astype(int)

print("Validated modelling dataset:", data.shape)
print("Unmatched records: 0")

## 4. Inspect the target and feature availability

The M4 target is imbalanced. Accuracy alone is therefore not sufficient to evaluate the classifier. We will report precision, recall, F1, balanced accuracy, average precision (PR-AUC summary) and the confusion matrix.

In [ ]:
target_counts = data["ml_target_conservative_hz"].value_counts().sort_index()
target_summary = pd.DataFrame({
    "count": target_counts,
    "proportion": data["ml_target_conservative_hz"].value_counts(normalize=True).sort_index()
})
target_summary.index = ["Negative (0)", "Positive (1)"]

display(target_summary)

availability = data[
    ["pl_rade", "pl_bmasse", "pl_bmassprov", "discoverymethod", "disc_year"]
].agg(["count", "isna"]).T
availability.columns = ["non_missing_count", "missing_count"]
availability["missing_percent"] = (
    availability["missing_count"] / len(data) * 100
).round(2)

display(availability)

## 5. Define the first feature set and leakage controls

### Initial predictors

- `pl_rade`: measured planetary radius
- `pl_bmasse`: reported planetary mass
- `pl_bmassprov`: mass provenance (important because `Msini` is a minimum mass)
- `discoverymethod`: discovery-method context
- `disc_year`: discovery-period context

### Excluded from this first experiment

We exclude `pl_insol`, `st_teff`, `pl_eqt`, orbital period, semi-major axis, HZ boundaries, HZ classifications and uncertainty-derived target fields. These variables either directly contribute to, or can closely proxy, the M4 target construction.

This is a conservative first experiment, not proof that all observational or selection biases have been removed. In particular, radius, mass and discovery method can still reflect discovery and measurement biases.

In [ ]:
TARGET = "ml_target_conservative_hz"

NUMERIC_FEATURES = ["pl_rade", "pl_bmasse", "disc_year"]
CATEGORICAL_FEATURES = ["pl_bmassprov", "discoverymethod"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

X = data[FEATURES].copy()
y = data[TARGET].copy()

print("Predictors:", FEATURES)
print("Target:", TARGET)
print("Rows:", len(X))
print("Positive labels:", int(y.sum()))
print("Negative labels:", int((y == 0).sum()))

# Guard against accidental inclusion of known target-construction variables.
forbidden_features = {
    "pl_insol", "st_teff", "pl_eqt", "pl_orbper", "pl_orbsmax",
    "hz_runaway_greenhouse_flux", "hz_maximum_greenhouse_flux",
    "hz_screening_category", "insol_uncertainty_assessment"
}
assert not (set(FEATURES) & forbidden_features), "Potential target leakage detected."

print("Leakage exclusion check passed for the explicitly listed variables.")

## 6. Stratified train/test split

The test set is held back until final evaluation. Preprocessing is placed inside the scikit-learn pipeline so that imputation, scaling and category encoding are fitted on training data only.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("\nTraining class counts:")
display(y_train.value_counts().sort_index().rename("count").to_frame())
print("\nTest class counts:")
display(y_test.value_counts().sort_index().rename("count").to_frame())

if y_test.nunique() != 2:
    raise ValueError("The test set must contain both target classes.")

## 7. Build preprocessing and models

We use:
- A **majority-class baseline** as the simple reference.
- **Logistic regression with class weighting** as the first trained classifier.

Class weighting addresses the imbalance during fitting, but it does not guarantee good minority-class performance. Evaluation will determine whether the model is useful.

In [ ]:
numeric_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_preprocessor, NUMERIC_FEATURES),
    ("categorical", categorical_preprocessor, CATEGORICAL_FEATURES)
])

baseline_model = DummyClassifier(strategy="most_frequent")
baseline_model.fit(X_train, y_train)

ml_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=RANDOM_STATE
    ))
])

ml_model.fit(X_train, y_train)

print("Baseline and logistic regression models trained.")

## 8. Evaluate on the held-out test set

The test set has not been used to fit the preprocessing steps or model. Interpret the model's results as performance at reproducing the M4 screening labels, not as performance at predicting habitability or life.

In [ ]:
def calculate_metrics(name, y_true, y_pred, y_probability=None):
    result = {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0)
    }
    if y_probability is not None:
        result["average_precision"] = average_precision_score(y_true, y_probability)
    return result

baseline_pred = baseline_model.predict(X_test)
ml_pred_050 = ml_model.predict(X_test)
ml_probability = ml_model.predict_proba(X_test)[:, 1]

comparison = pd.DataFrame([
    calculate_metrics("Majority-class baseline", y_test, baseline_pred),
    calculate_metrics("Logistic regression (0.50 threshold)", y_test, ml_pred_050, ml_probability)
]).set_index("model")

display(comparison.round(4))

print("Logistic regression classification report:")
print(classification_report(y_test, ml_pred_050, zero_division=0))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, ml_pred_050, labels=[0, 1],
    display_labels=["Negative", "Positive"],
    cmap="Blues", ax=ax, colorbar=False
)
ax.set_title("Logistic regression — held-out test set")
plt.tight_layout()
plt.show()

## 9. Block 4 — Decision threshold analysis

The threshold converts the model's estimated probability of the **M4 screening label** into a yes/no decision. It is not a probability of life.

We compare several predeclared thresholds on the same held-out test set to show the trade-off between false positives and false negatives. Do not choose the final threshold by maximising test-set performance; threshold selection should ultimately be done using validation data or cross-validation, then evaluated once on the untouched test set.

In [ ]:
thresholds = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]
threshold_rows = []

for threshold in thresholds:
    pred = (ml_probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()

    threshold_rows.append({
        "threshold": threshold,
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "true_negatives": tn
    })

threshold_results = pd.DataFrame(threshold_rows)
display(threshold_results.round(4))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(threshold_results["threshold"], threshold_results["precision"], marker="o", label="Precision")
ax.plot(threshold_results["threshold"], threshold_results["recall"], marker="o", label="Recall")
ax.plot(threshold_results["threshold"], threshold_results["f1"], marker="o", label="F1")
ax.set_xlabel("Decision threshold")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)
ax.set_title("Decision threshold trade-off")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 10. Inspect false positives and false negatives

These are errors relative to the M4 screening label. They are not confirmed scientific errors about whether a planet is habitable.

The selected threshold below is initially 0.50 for transparent reporting. It must not be described as an optimised threshold.

In [ ]:
INITIAL_THRESHOLD = 0.50
test_results = X_test.copy()
test_results["planet_name"] = data.loc[X_test.index, "pl_name"]
test_results["actual_m4_label"] = y_test
test_results["predicted_probability"] = ml_probability
test_results["predicted_m4_label"] = (
    ml_probability >= INITIAL_THRESHOLD
).astype(int)

false_positives = test_results[
    (test_results["actual_m4_label"] == 0) &
    (test_results["predicted_m4_label"] == 1)
].sort_values("predicted_probability", ascending=False)

false_negatives = test_results[
    (test_results["actual_m4_label"] == 1) &
    (test_results["predicted_m4_label"] == 0)
].sort_values("predicted_probability", ascending=True)

print("False positives:", len(false_positives))
display(false_positives.head(10))

print("False negatives:", len(false_negatives))
display(false_negatives.head(10))

## 11. Model interpretation

The logistic-regression coefficients describe associations with the model's predicted M4 screening label within this dataset. They do not establish causal relationships or biological habitability.

In [ ]:
fitted_preprocessor = ml_model.named_steps["preprocessor"]
feature_names = fitted_preprocessor.get_feature_names_out()
coefficients = ml_model.named_steps["classifier"].coef_[0]

coefficient_table = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})
coefficient_table["absolute_coefficient"] = coefficient_table["coefficient"].abs()
coefficient_table = coefficient_table.sort_values(
    "absolute_coefficient", ascending=False
)

display(coefficient_table.head(20).round(4))

## 12. M5 scientific interpretation and limitations

Complete this section after reviewing the actual results.

### What this experiment can conclude

- Whether the selected predictors contain signal for reproducing the M4 conservative HZ screening labels.
- Whether logistic regression improves on the majority-class baseline on held-out data.
- How the decision threshold changes precision, recall, false positives and false negatives.

### What this experiment cannot conclude

- It cannot establish that a planet is habitable.
- It cannot estimate the probability that life exists.
- It cannot establish surface liquid water, atmospheric composition, biosignatures or climate.
- The M4 labels are protocol-derived pseudo-labels, not independently verified ground truth.
- Results may be affected by class imbalance, measurement uncertainty, missing data, discovery bias and limited predictor information.

### M5 completion checklist

- [ ] M3/M4 join validated with no unmatched records.
- [ ] Target meaning and limitations documented.
- [ ] Leakage-conscious predictors justified.
- [ ] Stratified held-out test set created.
- [ ] Majority-class baseline evaluated.
- [ ] Trained model evaluated on unseen data.
- [ ] Precision, recall, F1, balanced accuracy and average precision reported.
- [ ] Confusion matrix reported.
- [ ] At least 20 labelled test cases evaluated for Block 4.
- [ ] Decision threshold and false-positive/false-negative trade-off documented.
- [ ] Results interpreted without claiming habitability or life prediction.

**Do not mark M5 complete until the notebook has been run and its results reviewed.**

## 13. Next steps

After the first run, review:
1. Whether the model beats the baseline on meaningful minority-class metrics.
2. Whether false positives and false negatives are scientifically interpretable.
3. Whether cross-validation and threshold selection are needed before finalising Block 4.
4. Whether the project should add a separate transparent habitability-potential assessment based on explicit scientific evidence. Such an assessment must not be presented as a probability of life.

M6 will integrate Qwen as the language-model component and decision judge. Qwen must explain evidence and limitations; it must not invent missing scientific measurements or override computed results.